# Lasso — Regression Shrinkage and Selection via the Lasso (1996)

_Papers · Classical ML_

**Add an L1 penalty (the sum of absolute coefficient values) to least squares, and the fit both shrinks and selects: it drives whole coefficients to exactly zero, doing variable selection automatically.**

---

This notebook is a practice scaffold for the **Lasso — Regression Shrinkage and Selection via the Lasso (1996)** lesson. We build the lasso up one piece at a time — the soft-thresholding operator, then coordinate descent, then the coefficient paths — running and reading each step before the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Define and check the soft-thresholding operator

The lasso's whole behaviour comes from one little function: **soft-thresholding**, $S(z, \gamma) = \operatorname{sign}(z)\,\max(|z| - \gamma,\, 0)$. It pulls every value toward zero by $\gamma$, and snaps anything already within $\gamma$ of zero to *exactly* zero — that snap is what produces variable selection.

We verify it against a value worked out by hand: $S([3,-2,0.5,-0.5,0],\,1) = [2,-1,0,0,0]$.

In [ ]:
import torch

torch.set_printoptions(precision=6, sci_mode=False)
torch.manual_seed(0)

# Soft-thresholding operator: S(z, g) = sign(z) * max(|z| - g, 0).
def soft_threshold(z, g):
    shrunk = torch.clamp(z.abs() - g, min=0.0)   # |z| - g, floored at zero
    return torch.sign(z) * shrunk                # keep the sign, apply the shrinkage

# Verify against a HAND-COMPUTED reference: S([3,-2,0.5,-0.5,0], 1) = [2,-1,0,0,0].
z = torch.tensor([3.0, -2.0, 0.5, -0.5, 0.0])
ref = torch.tensor([2.0, -1.0, 0.0, 0.0, 0.0])    # worked by hand in the lesson
mine = soft_threshold(z, 1.0)
print("soft-threshold:", mine.tolist())
print("allclose to hand reference [2,-1,0,0,0]:", torch.allclose(mine, ref, atol=1e-7))

### Step 2 — Solve the lasso by coordinate descent

Coordinate descent updates one coefficient at a time: hold the others fixed, compute the partial residual (the error left over without feature $j$), correlate feature $j$ with it, then soft-threshold that correlation. Cycling over all features many times converges to the lasso solution — no autograd needed.

On **orthonormal** predictors ($X^\top X / N = I$) the lasso has an exact closed form: just soft-threshold the OLS coefficients. We plant OLS coefficients $[4, -3, 0.7]$ and confirm coordinate descent matches that closed form, with the small $0.7$ coefficient selected out at threshold $1$.

In [ ]:
# Lasso by coordinate descent, from scratch (no nn, no autograd).
def lasso_cd(X, y, lam, n_iter=500):
    N, P = X.shape
    b = torch.zeros(P, dtype=X.dtype)
    col_ss = (X * X).sum(0) / N                   # feature squared length (= 1 if standardized)
    for _ in range(n_iter):
        for j in range(P):
            r_j = y - X @ b + X[:, j] * b[j]      # partial residual: error w/o feature j
            rho = (X[:, j] * r_j).sum() / N       # correlate feature j with it
            b[j] = soft_threshold(rho, lam) / col_ss[j]
    return b

# Orthonormal predictors so X^T X / N = I  ->  lasso has the EXACT closed form S(b_ols, lam).
N, P = 50, 3
Q, _ = torch.linalg.qr(torch.randn(N, P, dtype=torch.float64))
X = Q * (N ** 0.5)                                # columns orthonormal, X^T X / N = I
y = X @ torch.tensor([4.0, -3.0, 0.7], dtype=torch.float64)   # plant OLS coeffs [4, -3, 0.7]

b_ols = (X.t() @ y) / N                           # OLS = X^T y / N here
b_cd = lasso_cd(X, y, lam=1.0)                    # our from-scratch coordinate descent
b_closed = soft_threshold(b_ols, 1.0)             # closed form: soft-threshold the OLS coeffs
print("OLS coeffs        :", [round(v, 4) for v in b_ols.tolist()])
print("coordinate descent:", [round(v, 4) for v in b_cd.tolist()])
print("closed-form S(bols):", [round(v, 4) for v in b_closed.tolist()])
print("allclose CD vs closed form:", torch.allclose(b_cd, b_closed, atol=1e-8))
# -> our solver converges to [3, -2, 0]: the 0.7 coefficient (< threshold 1) is selected out.

### Step 3 — Trace the coefficient paths as lambda grows

Now the headline result. We build standardized data where only three of five features are real (coefficients $[3, -2, 0, 1.5, 0]$), then sweep the penalty strength $\lambda$ upward. Watch the two true-zero features get killed first, and the nonzero count fall $5 \to 3 \to 2 \to 1$ — the lasso selecting variables one by one as the budget tightens.

In [ ]:
# Coefficient paths: drive lambda up, watch coefficients hit EXACTLY zero.
torch.manual_seed(1)
Xv = torch.randn(40, 5, dtype=torch.float64)
Xv = (Xv - Xv.mean(0)) / Xv.std(0)                # standardize features
true_b = torch.tensor([3.0, -2.0, 0.0, 1.5, 0.0], dtype=torch.float64)  # only 3 are real
yv = Xv @ true_b + 0.1 * torch.randn(40, dtype=torch.float64)
yv = yv - yv.mean()

print("lambda |   b1      b2      b3      b4      b5   | nonzero")
for lam in [0.0, 0.1, 0.3, 0.6, 1.0, 1.5, 2.5]:
    b = lasso_cd(Xv, yv, lam, n_iter=2000)
    nz = int((b.abs() > 1e-8).sum())
    print(f"{lam:5.2f}  | " + " ".join(f"{v:7.3f}" for v in b.tolist()) + f"  |   {nz}")
# Our small run: the two true-zero features (b3, b5) are killed first at lam=0.1;
# nonzero count drops 5 -> 3 -> 2 -> 1 as lambda grows. This is OUR run, not the paper's number.

## Visualize the data & results

_As the L1 penalty strength (lambda) grows, do the lasso's coefficients shrink continuously to EXACTLY zero, one by one (variable selection)?_

### Step 1 — Rebuild the standardized regression data

This visualization block is self-contained, so we re-import torch and redefine the soft-thresholding operator and coordinate-descent solver, then regenerate the same standardized 5-feature dataset (only three features real). Keeping it standalone means you can run this section on its own.

In [ ]:
import torch

torch.manual_seed(1)

def soft_threshold(z, g):
    shrunk = torch.clamp(z.abs() - g, min=0.0)
    return torch.sign(z) * shrunk

def lasso_cd(X, y, lam, n_iter=2000):
    N, P = X.shape
    b = torch.zeros(P, dtype=X.dtype)
    css = (X * X).sum(0) / N
    for _ in range(n_iter):
        for j in range(P):
            r = y - X @ b + X[:, j] * b[j]
            rho = (X[:, j] * r).sum() / N
            b[j] = soft_threshold(rho, lam) / css[j]
    return b

X = torch.randn(40, 5, dtype=torch.float64)
X = (X - X.mean(0)) / X.std(0)                     # standardize
y = X @ torch.tensor([3., -2., 0., 1.5, 0.], dtype=torch.float64) + 0.1 * torch.randn(40, dtype=torch.float64)
y = y - y.mean()

### Step 2 — Sweep lambda and print the shrinking coefficient paths

We fit the lasso across the same $\lambda$ grid and print each coefficient vector plus its nonzero count. Reading down the rows, you can see every coefficient shrink smoothly toward zero and then snap to exactly zero — the continuous-shrinkage-plus-selection behaviour that is the paper's signature.

In [ ]:
for lam in [0.0, 0.1, 0.3, 0.6, 1.0, 1.5, 2.5]:
    b = lasso_cd(X, y, lam)
    coeffs = [round(v, 4) for v in b.tolist()]
    nonzero = int((b.abs() > 1e-8).sum())
    print(f"lam={lam:4.2f}  coeffs={coeffs}  nonzero={nonzero}")
# Our small run:
# lam=0.00  [3.0161,-1.9936,-0.0356,1.501,-0.0117]   nonzero=5
# lam=0.10  [2.9321,-1.8965,0.0,1.3881,0.0]          nonzero=3
# lam=0.30  [2.7672,-1.678,0.0,1.1715,0.0]           nonzero=3
# lam=0.60  [2.5199,-1.3504,0.0,0.8466,0.0]          nonzero=3
# lam=1.00  [2.1901,-0.9135,0.0,0.4134,0.0]          nonzero=3
# lam=1.50  [1.7637,-0.385,0.0,0.0,0.0]              nonzero=2
# lam=2.50  [0.7713,0.0,0.0,0.0,0.0]                 nonzero=1

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The L1-vs-L2 ablation. You have a working coordinate-descent lasso. Replace the L1 update
            (soft-thresholding) with the ridge (L2) update, which in the orthonormal case multiplies each OLS
            coefficient by $1/(1+\lambda)$ instead of soft-thresholding it. Fit both on the same standardized
            data across a grid of penalty strengths and count, for each method, how many coefficients are exactly
            zero. What do you expect, and what does it show about the paper's claim?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For each $\lambda$, run the lasso (soft-threshold update) and the ridge (uniform-shrink update) on the identical data. — _Holding the data and penalty grid fixed isolates the penalty shape (L1 vs L2) as the only variable._
- Count exact zeros (coefficients with $|\beta_j| \lt 10^{-8}$) for each method at each $\lambda$. — _Sparsity is the quantity the paper claims distinguishes the two: the lasso selects, ridge does not._
- Observe the lasso's zero-count rises as $\lambda$ grows, while ridge's stays at $0$ (no exact zeros, just smaller numbers). — _The L1 kink at the origin can pin coefficients at zero; the smooth L2 penalty has vanishing slope there and never does._

**Answer:** The lasso drives an increasing number of coefficients to exactly zero as $\lambda$
                 grows, while ridge keeps all of them nonzero (merely smaller). Because the only thing that
                 changed is the penalty shape &mdash; absolute value versus square &mdash; the difference isolates
                 L1's non-differentiable kink at the origin as the cause of selection. That is exactly the paper's
                 claim: the lasso shrinks and selects; ridge only shrinks. The CODEVIZ panel shows the
                 lasso's coefficient paths reaching zero one by one.

</details>

**Problem 2.** Apply soft-thresholding with $\gamma = 1$ to the OLS coefficient vector
            $\hat\beta^{\,\text{OLS}} = [4,\,-3,\,0.7]$ (orthonormal predictors), by hand. Which coefficient
            is selected out (set to zero), and what is the resulting lasso fit?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $S(4,1) = \operatorname{sign}(4)\,(|4|-1)_+ = 4-1 = 3$. — _Above the threshold, so shrunk toward zero by exactly $\gamma=1$._
- $S(-3,1) = \operatorname{sign}(-3)\,(|-3|-1)_+ = -(3-1) = -2$. — _Magnitude $3 \gt 1$, so it survives but is pulled toward zero by $1$; sign preserved._
- $S(0.7,1) = \operatorname{sign}(0.7)\,(|0.7|-1)_+ = (0.7-1)_+ = 0$. — _Magnitude $0.7 \lt 1$, so the positive part is zero &mdash; this coefficient is selected out._

**Answer:** The lasso fit is $[\,3,\,-2,\,0\,]$. The third coefficient ($0.7$, below the threshold
                 $1$) is set to exactly zero &mdash; selected out &mdash; while the other two shrink toward
                 zero by $1$. This is the verified converged solution: the notebook's coordinate descent reaches
                 this same vector from scratch and confirms it with torch.allclose.

</details>

**Problem 3.** In the penalized form, the lasso minimizes (residual sum of squares) $+\ \lambda\sum_j|\beta_j|$.
            What happens to the fitted coefficients as $\lambda \to 0$, and as $\lambda \to \infty$? Connect
            each limit to the budget $t$ in the constraint form.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- As $\lambda \to 0$, the penalty vanishes, so we minimize pure squared error. — _With no penalty the lasso reduces to ordinary least squares &mdash; every coefficient is its unpenalized OLS value._
- As $\lambda \to \infty$, the penalty dominates, so the cheapest objective sets all coefficients to zero. — _Any nonzero coefficient incurs an arbitrarily large penalty, so the minimizer is the all-zero vector._
- Match to $t$: small $\lambda$ = large budget $t$ (loose), large $\lambda$ = small budget $t$ (tight, eventually $t=0$). — _$\lambda$ and $t$ are inverse views of the same control._

**Answer:** As $\lambda \to 0$ the lasso becomes ordinary least squares (no shrinkage, all coefficients
                 at their OLS values &mdash; equivalent to a very large budget $t$). As $\lambda \to \infty$ all
                 coefficients are driven to exactly zero (equivalent to budget $t=0$). In between, raising
                 $\lambda$ (tightening $t$) zeroes coefficients one at a time, tracing the sparse path the paper
                 is famous for.

</details>